# padelpy2 quickstart tutorial

This notebook shows how to compute **stock Yap PaDEL-Descriptor** values from RDKit molecules with padelpy2: a minimal `Weight` calculation, a note on aromatic molecules (stock JAR vs `detectaromaticity`), and a small custom descriptor subset. You will leave knowing when to use padelpy2 versus padelpy.

## Setup

Requirements: Python 3.9+, **Java JRE 8+** on `PATH`, RDKit (conda-forge recommended), and `padelpy2` (`pip install padelpy2` or an editable checkout). See the Sphinx installation page for details.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

import padelpy2
from padelpy2 import Calculator, PaDELConfig
from padelpy2.descriptors import ALOGP, AromaticAtomsCount, AromaticBondsCount, Crippen, Weight

print("padelpy2", padelpy2.__version__)
CFG = PaDELConfig(threads=1)

## Motivation and background

Many QSAR workflows still expect columns from the classic PaDEL-Descriptor Java application (Yap, 2011). padelpy offers a thin stdlib CLI over that **stock JAR**. padelpy2 adds RDKit → DataFrame workflows while keeping stock-JAR semantics and a padelpy-compatible `padeldescriptor` surface.

Engine identity: Yap, C. W. (2011). PaDEL-Descriptor: An open source software to calculate molecular descriptors and fingerprints. *Journal of Computational Chemistry*, 32(7), 1466–1474. https://doi.org/10.1002/jcc.21707

## Minimal example

Build two molecules (ethanol and benzene) and compute the `Weight` descriptor class.

In [ ]:
def mols_from_smiles(smiles_list):
    mols = []
    for smi in smiles_list:
        mol = Chem.AddHs(Chem.MolFromSmiles(smi))
        AllChem.Compute2DCoords(mol)
        mols.append(mol)
    return mols

smiles = ["CCO", "c1ccccc1"]
mols = mols_from_smiles(smiles)
df_weight = Calculator([Weight], config=CFG)(mols)
df_weight

By default the engine `Name` column is dropped. Default full-catalog shapes (1444 / 431 / 1875) are documented under API stability in the Sphinx docs.

## Progressive deep dive: aromatic molecules

With default `PaDELConfig` (`detectaromaticity=False`), the stock JAR often reports `naAromAtom = 0` and `nAromBond = 0` for benzene. Enabling `detectaromaticity=True` changes those counts. Defaults are part of the stock-JAR fidelity contract (see the project when-to-use docs).

In [ ]:
benzene = mols_from_smiles(["c1ccccc1"])
arom = [AromaticAtomsCount, AromaticBondsCount]

off = Calculator(arom, config=PaDELConfig(threads=1))(benzene)
on = Calculator(
    arom, config=PaDELConfig(threads=1, detectaromaticity=True)
)(benzene)

print("default config:\n", off)
print("detectaromaticity=True:\n", on)

## Progressive deep dive: custom subset

Mix descriptor classes freely. Here ALOGP, Crippen, and Weight match a common small subset used in stock-JAR checks.

In [ ]:
subset = Calculator([ALOGP, Crippen, Weight], config=CFG)(mols)
print(subset.shape)
subset

## Results

The tables above are the results: `Weight` columns for two molecules, aromatic counts under two configs, and a seven-column physicochemical subset. Look for (1) finite numeric values for ethanol/benzene weights, (2) aromatic counts flipping from 0/0 to 6/6 when detection is enabled, and (3) subset shape `(2, 7)`.

## Interpretation / discussion

Stock-JAR defaults are a deliberate fidelity contract, not a bug. If you need stdlib-only dict helpers, stay on padelpy. padelpy2 is for RDKit/DataFrame workflows that must match classic PaDEL columns.

## Takeaways

- Use `Calculator([…])` with RDKit `Mol` objects; results are `pandas` DataFrames with `Name` dropped by default.
- Prefer small custom subsets while exploring; full 2D catalogs are large and slower.
- For aromatic molecules, decide explicitly whether to set `detectaromaticity=True` under stock JAR defaults.
- Choose padelpy (stdlib) or padelpy2 (stock JAR + RDKit) using the project when-to-use guide.
- Java must be on `PATH`; padelpy2 does not download a JRE.

## Further reading

- Sphinx docs: installation, quickstart, when-to-use, migration from padelpy, API stability
- Repository: `README.md`, `examples/example.ipynb` (this file)
- Yap, C. W. (2011). PaDEL-Descriptor. *J. Comput. Chem.* https://doi.org/10.1002/jcc.21707
- padelpy: https://github.com/ecrl/padelpy
